In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
import shap
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
columns = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes",
    "land","wrong_fragment","urgent","hot","num_failed_logins","logged_in",
    "num_compromised","root_shell","su_attempted","num_root","num_file_creations",
    "num_shells","num_access_files","num_outbound_cmds","is_host_login",
    "is_guest_login","count","srv_count","serror_rate","srv_serror_rate",
    "rerror_rate","srv_rerror_rate","same_srv_rate","diff_srv_rate",
    "srv_diff_host_rate","dst_host_count","dst_host_srv_count",
    "dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate",
    "dst_host_rerror_rate","dst_host_srv_rerror_rate","label","difficulty"
]

drop_cols = [
    "difficulty",
    "srv_serror_rate", "dst_host_srv_serror_rate",
    "srv_rerror_rate", "dst_host_srv_rerror_rate",
]
categorical_cols = ["protocol_type", "service", "flag"]

In [3]:
family_map = {
    "neptune": "DoS", "smurf": "DoS", "pod": "DoS", "teardrop": "DoS",
    "land": "DoS", "back": "DoS", "apache2": "DoS", "udpstorm": "DoS",
    "processtable": "DoS", "mailbomb": "DoS",
    "ipsweep": "Probe", "portsweep": "Probe", "satan": "Probe",
    "nmap": "Probe", "mscan": "Probe", "saint": "Probe",
    "guess_passwd": "R2L", "ftp_write": "R2L", "imap": "R2L",
    "warezmaster": "R2L", "spy": "R2L", "phf": "R2L", "multihop": "R2L",
    "warezclient": "R2L", "sendmail": "R2L", "named": "R2L",
    "snmpgetattack": "R2L", "snmpguess": "R2L", "httptunnel": "R2L",
    "xlock": "R2L", "xsnoop": "R2L", "worm": "R2L",
    "buffer_overflow": "U2R", "loadmodule": "U2R", "rootkit": "U2R",
    "perl": "U2R", "sqlattack": "U2R", "xterm": "U2R", "ps": "U2R",
}

def map_family(label):
    if label == "normal":
        return "normal"
    return family_map.get(label, "other")

In [4]:
train_df = pd.read_csv("../data/KDDTrain.txt", names=columns)
test_df  = pd.read_csv("../data/KDDTest.txt",  names=columns)

train_raw_labels = train_df["label"].copy()
test_raw_labels  = test_df["label"].copy()

train_df["family"] = train_df["label"].apply(map_family)
test_df["family"]  = test_df["label"].apply(map_family)

train_df = train_df.drop(columns=drop_cols + ["label"])
test_df  = test_df.drop(columns=drop_cols  + ["label"])

In [5]:
for col in categorical_cols:
    le = LabelEncoder()
    combined = pd.concat([train_df[col], test_df[col]], axis=0)
    le.fit(combined)
    train_df[col] = le.transform(train_df[col])
    test_df[col]  = le.transform(test_df[col])

In [6]:
X_train = train_df.drop(columns=["family"])
X_test  = test_df.drop(columns=["family"])
y_train_raw = train_df["family"]
y_test_raw  = test_df["family"]

In [7]:
family_encoder = LabelEncoder()
family_encoder.fit(pd.concat([y_train_raw, y_test_raw]))
y_train = family_encoder.transform(y_train_raw)
y_test  = family_encoder.transform(y_test_raw)

In [8]:
n_classes = len(family_encoder.classes_)
class_counts = pd.Series(y_train).value_counts().sort_index()
class_weights = len(y_train) / (n_classes * class_counts)
sample_weights = np.array([class_weights[y] for y in y_train])

In [9]:
model = XGBClassifier(
    objective="multi:softmax",
    num_class=n_classes,
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric="mlogloss",
    n_jobs=-1
)

model.fit(X_train, y_train, sample_weight=sample_weights)

,objective,'multi:softmax'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,'mlogloss'


In [10]:
explainer = shap.TreeExplainer(model)

sample_indices = []
for family in family_encoder.classes_:
    idx = X_test[y_test_raw == family].index
    n   = min(200, len(idx))
    sample_indices.extend(idx[:n].tolist())

X_sample     = X_test.loc[sample_indices]
y_sample_raw = y_test_raw.loc[sample_indices]

print("Sample sizes per family:")
for fam in family_encoder.classes_:
    print(f"  {fam}: {(y_sample_raw==fam).sum()}")

Sample sizes per family:
  DoS: 200
  Probe: 200
  R2L: 200
  U2R: 67
  normal: 200


In [11]:
shap_values_raw = explainer.shap_values(X_sample)
if isinstance(shap_values_raw, list):
    shap_values_list = shap_values_raw
else:
    shap_values_list = [shap_values_raw[:, :, i] for i in range(shap_values_raw.shape[2])]

print(f"shap_values_list length: {len(shap_values_list)}")
print(f"Each array shape: {shap_values_list[0].shape}")

shap_values_list length: 5
Each array shape: (867, 37)


In [12]:
for i, family in enumerate(family_encoder.classes_):
    plt.figure()
    shap.summary_plot(shap_values_list[i], X_sample, show=False, max_display=15, plot_size=(10, 7))
    plt.title(f"SHAP Features pushing toward '{family}' prediction")
    plt.tight_layout()
    plt.savefig(f"mc_shap_{family}.png", dpi=120, bbox_inches="tight")
    plt.close()

In [13]:
mean_abs_all = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values_list], axis=0)
top15_idx    = np.argsort(mean_abs_all)[::-1][:15]
top15_names  = X_train.columns[top15_idx]

heatmap_data = pd.DataFrame(index=top15_names, columns=family_encoder.classes_, dtype=float)
for i, family in enumerate(family_encoder.classes_):
    heatmap_data[family] = shap_values_list[i][:, top15_idx].mean(axis=0)

In [14]:
plt.figure(figsize=(10, 8))
sns.heatmap(heatmap_data.astype(float), cmap="coolwarm", center=0, annot=True, fmt=".2f", linewidths=0.5)
plt.title("Mean SHAP per Feature per Family")
plt.xlabel("Attack Family")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig("mc_shap_heatmap.png", dpi=120, bbox_inches="tight")
plt.close()

In [15]:
y_pred     = model.predict(X_test)
y_pred_fam = family_encoder.inverse_transform(y_pred)
y_test_fam = y_test_raw.values

base_values = explainer.expected_value

for i, family in enumerate(family_encoder.classes_):
    correct_mask = (y_test_fam == family) & (y_pred_fam == family)
    correct_idx  = X_test.index[correct_mask]

    if len(correct_idx) == 0:
        print(f"{family}: no correctly predicted examples, skipping")
        continue

    row_idx = correct_idx[0]
    row     = X_test.loc[[row_idx]]
    sv_raw  = explainer.shap_values(row)
    sv      = sv_raw if isinstance(sv_raw, list) else [sv_raw[:, :, c] for c in range(sv_raw.shape[2])]

    explanation = shap.Explanation(
        values=sv[i][0],
        base_values=base_values[i],
        data=row.iloc[0].values,
        feature_names=list(X_test.columns)
    )

    plt.figure(figsize=(10, 7))
    shap.plots.waterfall(explanation, show=False, max_display=12)
    plt.title(f"Waterfall {family} (correctly predicted)")
    plt.tight_layout()
    plt.savefig(f"mc_waterfall_{family}.png", dpi=120, bbox_inches="tight")
    plt.close()